In [8]:
import numpy as np
import tensorflow as tf

In [46]:
def train_logreg(X: np.ndarray, y: np.ndarray, learning_rate: float, iterations: int) -> tuple[list[float], ...]:
    """
    Gradient-descent training algorithm for logistic regression, optimizing parameters with Binary Cross Entropy loss.
    """
    # key result: alpha loss / alpha beta = X^T (y_hat - y)
    y = y.reshape(-1,1) # make sure it represents a vector
    n, m = X.shape
    X = np.concatenate((np.ones_like(y), X), axis=1)
    w = np.zeros(shape=(m+1,1))
    count = 0
    loss = []
    while count < iterations:
        y_hat = 1 / (1 + np.exp(-np.matmul(X,w))) # update prediction
        log_likelihood = np.dot(y.T, np.log(y_hat)) + np.dot(1 - y.T, np.log(1 - y_hat))
        loss.append(-log_likelihood)
        w = w - learning_rate * np.matmul(X.T, (y_hat - y))  # update parameters, gradient step with learning rate
        count += 1
    return np.round(w,4), np.round(loss,4)

In [47]:
X = np.array([[1.0, 0.5], [-0.5, -1.5], [2.0, 1.5], [-2.0, -1.0]])
y = np.array([1, 0, 1, 0])
learning_rate = 0.01
iterations = 20

In [50]:
w, loss = train_logreg(X, y, learning_rate, iterations)

In [ ]:
# we could also just use tf for auto diff
def train_logreg_auto_diff(X: np.ndarray, y: np.ndarray, learning_rate: float, iterations: int) -> tuple[list[float], ...]:
    """
    Gradient-descent training algorithm for logistic regression, optimizing parameters with Binary Cross Entropy loss.
    """
    # key result: alpha loss / alpha beta = X^T (y_hat - y)
    y = y.reshape(-1,1)
    n, m = X.shape
    X_ones = np.ones(shape=(n,1))
    X = np.hstack((X_ones, X))
    w = np.zeros(shape=(m+1,1))

    X = tf.constant(X, dtype=tf.float32)
    y = tf.constant(y, dtype=tf.float32)
    w = tf.Variable(w, dtype=tf.float32)
    
    count = 0
    loss = []
    while count < iterations:
        with tf.GradientTape() as tape:
            y_hat = tf.sigmoid(tf.matmul(X,w))
            loss = - (tf.matmul(tf.transpose(y), tf.math.log(y_hat)) + tf.matmul(1 - tf.transpose(y), tf.math.log(1 - y_hat)))
        grad = tape.gradient(loss, w)
        w.assign(w - learning_rate*grad)
        count += 1
    
    return w

In [51]:
X = np.array([[1.0, 0.5], [-0.5, -1.5], [2.0, 1.5], [-2.0, -1.0]])
y = np.array([1, 0, 1, 0])
learning_rate = 0.01
iterations = 20

In [111]:
train_logreg_auto_diff(X,y,learning_rate, iterations)

<tf.Variable 'Variable:0' shape=(3, 1) dtype=float32, numpy=
array([[-3.1065999e-04],
       [ 4.0383106e-01],
       [ 3.3788186e-01]], dtype=float32)>